# Methodology Guide## St. Paul Neighborhood Health AnalysisDetailed technical guidance for data preparation, aggregation, indicator calculation, and analysis.

## Part 1: Understanding Your Datasets### Building Permits**Typical columns:**- Permit ID / Issue ID- Issue Date / Permit Date- Neighborhood / Ward / District- Address / Location- Permit Type (Residential, Commercial, etc.)- Estimated Cost- Status- Coordinates (Latitude/Longitude)**Analysis approach:**- Annual permit count by neighborhood- Average permit value- Mix of permit types- Development velocity (year-over-year growth)**Health indicator mapping:**- High permits + rising trend = Positive investment- Low permits + declining = Neighborhood decline- Residential bias = Housing-focused growth

### Crime Incidents**Typical columns:**- Incident ID- Report Date / Incident Date- Offense Type / Category- Crime Classification (Violent/Property/Other)- Neighborhood / Ward- Address / Coordinates- Status**Analysis approach:**- Total crimes per neighborhood per year- Violent vs. property crimes- Crime rate (per 1,000 residents)- Temporal trends**Health indicator mapping:**- Low & stable crime = High health- Rising crime = Declining health- Violent crime concentration = Safety concern- Property crime = Economic distress

### Housing Production (GeoJSON)**Typical structure:**- GeoJSON FeatureCollection- Each feature has geometry (Point/Polygon) and properties- Properties include: address, units, year, type, neighborhood**Analysis approach:**- Total units per neighborhood per year- Average units per project- Geographic clustering- Housing type breakdown**Health indicator mapping:**- Increasing units = Housing supply improvement- Stagnant production = Supply constraints- Mixed-income development = Healthy diversity

### Service Requests**Typical columns:**- Request ID / Case Number- Date / Request Date- Issue Type / Category- Status- Neighborhood / Ward- Address / Coordinates- Description- Resolution Date (optional)- Resolution Time**Analysis approach:**- Request volume per neighborhood per year- Issue type breakdown- Resolution times- Repeat issues**Health indicator mapping:**- High volume + poor resolution = Infrastructure neglect- Growing requests = Emerging problems- Infrastructure issues = Maintenance burden

## Part 2: Aggregation Strategy### Step 1: Create Neighborhood-Year SummariesFor each neighborhood-year combination, calculate:

In [ ]:
# PERMITS aggregation"""For each neighborhood-year:- total_permits: Count of all permits- permit_cost_total: Sum of estimated costs- residential_permits: Count of residential permits- commercial_permits: Count of commercial permits- avg_permit_cost: Mean cost per permit"""# Example code:# permits_agg = perms_clean.groupby(['NEIGHBORHOOD_STANDARD', 'YEAR']).agg({#     'PERMIT_ID': 'count',#     'ESTIMATED_COST': ['sum', 'mean'],#     'PERMIT_TYPE': lambda x: (x == 'Residential').sum()# }).rename(columns={'PERMIT_ID': 'total_permits'})print("Permits aggregation structure defined")print("See code cells below for implementation")

In [ ]:
# CRIME aggregation"""For each neighborhood-year:- total_crimes: Count of all incidents- violent_crimes: Count of violent crimes- property_crimes: Count of property crimes- other_crimes: Count of other crimes- crime_rate_per_1000: (total_crimes / population) * 1000"""# Example code:# crime_agg = crime_clean.groupby(['NEIGHBORHOOD_STANDARD', 'YEAR']).agg({#     'INCIDENT_ID': 'count',#     'CRIME_TYPE': lambda x: (x == 'VIOLENT').sum(),# }).rename(columns={'INCIDENT_ID': 'total_crimes'})print("Crime aggregation structure defined")print("See implementation below")

In [ ]:
# SERVICE REQUESTS aggregation"""For each neighborhood-year:- total_requests: Count of all requests- avg_resolution_days: Average time to close (if available)- closure_rate: % of closed requests- top_issue_type: Most common issue"""# Example code:# reqs_agg = requests_clean.groupby(['NEIGHBORHOOD_STANDARD', 'YEAR']).agg({#     'REQUEST_ID': 'count',#     'ISSUE_TYPE': 'nunique'# }).rename(columns={'REQUEST_ID': 'total_requests'})print("Service requests aggregation structure defined")

### Step 2: Combine Into Master Dataframe

In [ ]:
# Pseudo-code for joining aggregates"""master = permits_agg.join(crime_agg).join(requests_agg)master = master.reset_index()# Result structure:# master has columns:# - NEIGHBORHOOD_STANDARD# - YEAR# - total_permits, permit_cost_total, residential_permits, ...# - total_crimes, violent_crimes, property_crimes, ...# - total_requests, avg_resolution_days, ..."""print("Master dataframe structure:")print("Columns: NEIGHBORHOOD_STANDARD, YEAR, all aggregated metrics")print("Ready for indicator calculation")

## Part 3: Indicator Calculation

### Safety Index FormulaRange: 0-100 (higher = safer)**Step 1: Calculate crime rate**```Crime_Rate = (Total_Crimes / Population) × 1000```**Step 2: Normalize to 0-100 scale**```Crime_Rate_Scaled = (Crime_Rate - City_Min) / (City_Max - City_Min) × 100```**Step 3: Invert (lower crime = higher safety)**```Safety_Index = 100 - Crime_Rate_Scaled```**Step 4: Apply trend bonus/penalty**```If crime improving (declining): +5 bonusIf crime worsening (rising >5%): -10 penaltyFinal Safety_Index = Safety_Index + Trend```**Step 5: Clamp to 0-100**```Safety_Index = MAX(0, MIN(100, Safety_Index))```

### Development Index FormulaRange: 0-100 (higher = more active development)**Step 1: Normalize component metrics**- Permits per capita (scaled 0-100)- Housing units per capita (scaled 0-100)**Step 2: Combine**```Development_Index = (Permits_Scaled × 0.5) +                     (Housing_Scaled × 0.5)```**Step 3: Interpret**- 70+ = Active development- 30-70 = Moderate activity- <30 = Stagnant market

### Service Need Index FormulaRange: 0-100 (higher = more requests/needs)**Step 1: Calculate request rate**```Request_Rate = (Total_Requests / Population) × 1000```**Step 2: Normalize**```Request_Rate_Scaled = (Request_Rate - Min) / (Max - Min) × 100```**Step 3: Use as need indicator**```Service_Need_Index = Request_Rate_Scaled```**Step 4: Apply penalty for resolution time (if available)**```If Avg_Resolution_Days > City_Average:  Service_Need_Index = Service_Need_Index + 5```

### Composite Health Score FormulaRange: 0-100 (higher = healthier neighborhood)```Health_Score = (Safety_Index × 0.40) +                (Development_Index × 0.30) +                (100 - Service_Need_Index × 0.30)```**Weighting rationale:**- Safety is primary concern (40%)- Development is positive signal (30%)- Service need is inverse (30%)**Categories:**- 70-100: Excellent (healthy, safe, developing)- 50-70: Good (moderate conditions)- 30-50: Needs Attention (challenges)- 0-30: Declining (urgent issues)

## Part 4: Statistical Analysis### Correlation Analysis- Measure relationships between metrics- Pearson correlation for linear relationships- Spearman rank for non-linear- Visualize with heatmaps### Trend Analysis- Calculate year-over-year changes- Fit linear trend lines- Identify acceleration/deceleration- Compare neighborhoods### Clustering- K-means clustering of neighborhoods- Identify similar neighborhood groups- Interpret cluster characteristics- Visualize in 2D space### Hypothesis Testing- T-tests comparing neighborhood groups- ANOVA for multi-group comparisons- Chi-square for categorical associations

## Part 5: Quality Assurance Checklist### Data Validation- [ ] All neighborhoods in source data mapped to standard names- [ ] No duplicates in aggregated data- [ ] All percentages/rates verified by spot-checking- [ ] Time ranges documented for each dataset### Indicator Validation- [ ] Health scores between 0-100- [ ] Component indices make logical sense- [ ] Rankings are plausible- [ ] Spot-check 5-10 neighborhoods manually### Analysis Validation- [ ] Correlations seem reasonable- [ ] Trends match visual inspection- [ ] Clusters are interpretable- [ ] Results are reproducible### Documentation- [ ] Methodology fully documented- [ ] Exclusions and assumptions noted- [ ] Data limitations identified- [ ] Code is commented and clean

In [ ]:
# Quality assurance checklist codeqa_checklist = {    'data_validation': {        'neighborhoods_mapped': False,        'no_duplicates': False,        'calculations_verified': False,        'dates_documented': False    },    'indicator_validation': {        'scores_in_range': False,        'logic_sensible': False,        'rankings_plausible': False,        'spot_checks_pass': False    },    'analysis_validation': {        'correlations_reasonable': False,        'trends_match_visual': False,        'clusters_interpretable': False,        'reproducible': False    },    'documentation': {        'methodology_complete': False,        'exclusions_noted': False,        'limitations_identified': False,        'code_commented': False    }}import jsonprint("QA Checklist:")print(json.dumps(qa_checklist, indent=2))